# Regresión Logística — Riesgo Operativo

**SI3015 - Fundamentos de Aprendizaje Automático**

En este cuaderno entrenaremos una regresión logística con regularización L2 y con características polinómicas, usando validación cruzada para seleccionar buenos hiperparámetros. El objetivo es **clasificar** si un equipo tiene riesgo operativo alto o bajo (`riesgo_alto`) a partir de las variables del dataset.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import reciprocal
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, ConfusionMatrixDisplay, confusion_matrix

In [ ]:
# Definamos el "random_state" para que los resultados sean reproducibles:
random_state = 42

In [ ]:
# Cambiemos la fuente de las gráficas de matplotlib:
plt.rc('font', family='serif', size=12)

In [ ]:
# Carguemos el dataset de riesgo operativo (ya limpio):
df = pd.read_csv('dataset_riesgo_operativo_limpio.csv')

# Usamos dos features para poder visualizar la frontera de decisión en 2D:
# t_prom y pct_cumplimiento
X = df[['t_prom', 'pct_cumplimiento']].values
y = df['riesgo_alto'].values   # 1 = riesgo alto, 0 = riesgo bajo

print(f'X shape: {X.shape}')
print(f'Distribución de clases: {dict(zip(*np.unique(y, return_counts=True)))}')

In [ ]:
# Grafiquemos el dataset:
fig, ax = plt.subplots()
ax.scatter(X[:, 0], X[:, 1], s=20, c=np.where(y == 0, 'r', 'b'))
ax.set_xlabel('t_prom')
ax.set_ylabel('pct_cumplimiento')
ax.set_title('Riesgo bajo (rojo) vs Riesgo alto (azul)')
fig.set_size_inches(1.6*5, 5)

In [ ]:
# Separemos nuestros datos en conjuntos de entrenamiento y prueba:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)

In [ ]:
# Función auxiliar para graficar la frontera de decisión:

def plot_binary_classifcation(X, y, classifier=None, contour_alpha=0.1):

    def compute_predictions(X, classifier):
        return classifier.predict(X)

    cmap = ListedColormap(['#FF0000', '#0000FF'])
    x1_min, x1_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    x2_min, x2_max = X[:, 1].min() - 0.05, X[:, 1].max() + 0.05

    xx1, xx2 = np.meshgrid(
        np.arange(x1_min, x1_max, 0.05), np.arange(x2_min, x2_max, 0.005)
    )

    fig, ax = plt.subplots()

    if classifier is not None:
        X_ = np.array([xx1.ravel(), xx2.ravel()]).T
        z = compute_predictions(X_, classifier)
        z = np.reshape(z, xx1.shape)
        ax.contourf(xx1, xx2, z, alpha=contour_alpha, cmap=cmap)

    ax.set_xlim(xx1.min(), xx1.max())
    ax.set_ylim(xx2.min(), xx2.max())
    ax.scatter(X[:,0], X[:,1], c=y, cmap=cmap)
    ax.set_xlabel('t_prom')
    ax.set_ylabel('pct_cumplimiento')
    fig.set_size_inches(1.6*5, 5)

In [ ]:
# Definamos un pipeline de scikit-learn con nuestro modelo base:
lr_base = Pipeline([
    ('poly', PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression())
])

In [ ]:
# Definamos las distribuciones de parámetros sobre las que haremos la búsqueda:
param_distributions = {
    'poly__degree': list(range(2, 8)),
    'classifier__C': reciprocal(1e-5, 1e5)
}

In [ ]:
# Definamos nuestro modelo mediante RandomizedSearchCV:
lr = RandomizedSearchCV(
    lr_base,
    cv=4,
    param_distributions=param_distributions,
    n_iter=200,
    random_state=random_state
)

In [ ]:
# Entrenemos el modelo:
lr.fit(X_train, y_train)

In [ ]:
# Obtengamos los mejores hiperparámetros encontrados para el modelo:
lr.best_params_

In [ ]:
# Obtengamos la accuracy y el f1-score de prueba:
print(f'Accuracy: {lr.score(X_test, y_test)}')
print(f'F1 score: {f1_score(y_test, lr.predict(X_test))}')

In [ ]:
# Grafiquemos los datos junto con la frontera de decisión del modelo:
plot_binary_classifcation(X, y, classifier=lr)

In [ ]:
# Grafiquemos la matriz de confusión de los datos de prueba:
cm = confusion_matrix(y_test, lr.predict(X_test))
disp = ConfusionMatrixDisplay(cm)
disp.plot()
plt.show()